# M4 Assignment2 - Green Patent Detection (PatentSBERTa): Active Learning + LLM→Human HITL
>Part B: Identify High-Risk Examples (Uncertainty Sampling)

Suchanya Baiyam Business Data Science AAU


* Dataset:AI-Growth-Lab/patents_claims_1.5m_traim_test (The provided 50k samples)
* Model: AI-Growth-Lab/PatentSBERTa
* Working file: patents_50k_green.parquet with splits train_silver, pool_unlabeled, eval_silver
* Silver label: is_green_silver (derived from CPC Y02*)

Task
- Compute p_green for all examples in pool_unlabeled.
- Compute u for each example, select the top 100 highest-u rows.
- Export hitl_green_100.csv including: doc_id, text, p_green, u, and empty labeling columns.

In [1]:
!pip install sentence_transformers scikit-learn pyarrow -q

STEP01: Load Libraries

In [2]:
import pandas as pd
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer

STEP02: Load dataset and baseline model from Part A

In [3]:
df = pd.read_parquet("patent_50k_green.parquet")
pool_df = df[df["split"] == "pool_unlabeled"].copy()
print(pool_df.shape)

(10000, 4)


In [4]:
with open("baseline_logreg.pkl", "rb") as f:
  clf = pickle.load(f)

STEP03: Load model

In [5]:
model = SentenceTransformer("AI-Growth-Lab/PatentSBERTa")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

STEP04: Encode pool texts

In [6]:
x_pool = model.encode(pool_df["text"].tolist(), batch_size = 64, convert_to_numpy=True, show_progress_bar=True)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

STEP05: Compute p_green

In [7]:
probs = clf.predict_proba(x_pool)[:,1]
pool_df["p_green"] = probs

STEP06: Compute uncertainly score

u = 1 − 2 · |p − 0.5|

u = 1 means most uncertain (p = 0.5). u ≈ 0 means confident (p ≈ 0 or p ≈ 1).

In [8]:
pool_df["u"] = 1 - 2 * np.abs(pool_df["p_green"] - 0.5)

STEP07: Select top 100 high-risk

In [9]:
hitl_df = pool_df.sort_values("u", ascending=False).head(100)

STEP08: Prepare Human-In-The-Loop
- As instructed in Part C

In [10]:
hitl_df = hitl_df[["doc_id", "text", "p_green", "u"]].copy()

hitl_df["llm_green_suggested"] = ""
hitl_df["llm_confidence"] = ""
hitl_df["llm_rationale"] = ""
hitl_df["is_green_human"] = ""
hitl_df["notes"] = ""

STEP09: Export CSV

In [11]:
hitl_df.to_csv("hitl_green_100.csv", index=False)